# Qwen2-Audio baseline and int8 quantization
Starter notebook.


In [1]:
from pathlib import Path
import shutil
import csv
import pandas as pd
import torch

PROJECT_ROOT = Path("/home/paperspace/projects/iwslt2026-compression")
RAW_ROOT = PROJECT_ROOT / "data" / "raw" / "acl60_60" / "2" / "acl_6060"
FILTERED_ROOT = PROJECT_ROOT / "data" / "raw" / "acl6060_en_zh"
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"

DEV_MANIFEST = MANIFEST_DIR / "acl6060_dev_en_zh.csv"
EVAL_MANIFEST = MANIFEST_DIR / "acl6060_eval_en_zh.csv"

print("project root exists:", PROJECT_ROOT.exists())
print("raw root exists:", RAW_ROOT.exists())
print("manifest dir exists:", MANIFEST_DIR.exists())
print("dev manifest exists:", DEV_MANIFEST.exists())
print("eval manifest exists:", EVAL_MANIFEST.exists())
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

if not DEV_MANIFEST.exists() or not EVAL_MANIFEST.exists():
    print("Manifests missing. Rebuilding en->zh subset and manifests...")

    if FILTERED_ROOT.exists():
        shutil.rmtree(FILTERED_ROOT)

    for split in ["dev", "eval"]:
        (FILTERED_ROOT / split / "text" / "txt").mkdir(parents=True, exist_ok=True)
        (FILTERED_ROOT / split / "segmented_wavs").mkdir(parents=True, exist_ok=True)

        shutil.copy2(
            RAW_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt",
            FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt",
        )
        shutil.copy2(
            RAW_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt",
            FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt",
        )
        shutil.copytree(
            RAW_ROOT / split / "segmented_wavs" / "gold",
            FILTERED_ROOT / split / "segmented_wavs" / "gold",
        )

    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)

    for split in ["dev", "eval"]:
        en_file = FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.en.txt"
        zh_file = FILTERED_ROOT / split / "text" / "txt" / f"ACL.6060.{split}.en-xx.zh.txt"
        wav_dir = FILTERED_ROOT / split / "segmented_wavs" / "gold"

        en_lines = en_file.read_text(encoding="utf-8").splitlines()
        zh_lines = zh_file.read_text(encoding="utf-8").splitlines()
        wavs = sorted(wav_dir.glob("sent_*.wav"), key=lambda p: int(p.stem.split("_")[1]))

        assert len(en_lines) == len(zh_lines) == len(wavs), (split, len(en_lines), len(zh_lines), len(wavs))

        out_path = MANIFEST_DIR / f"acl6060_{split}_en_zh.csv"
        with out_path.open("w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["id", "split", "audio_path", "source_en", "target_zh"])
            for i, (wav, en, zh) in enumerate(zip(wavs, en_lines, zh_lines), start=1):
                writer.writerow([f"{split}_{i}", split, str(wav.relative_to(PROJECT_ROOT)), en, zh])

        print("wrote", out_path, "rows:", len(wavs))

df = pd.concat(
    [pd.read_csv(DEV_MANIFEST), pd.read_csv(EVAL_MANIFEST)],
    ignore_index=True
)

print("rows:", len(df))
df.head()

project root exists: True
raw root exists: True
manifest dir exists: True
dev manifest exists: True
eval manifest exists: True
cuda: True
gpu: NVIDIA RTX A6000
rows: 884


,id,split,audio_path,source_en,target_zh
0,dev_1,dev,data/raw/acl6060_en_zh/dev/segmented_wavs/gold...,"Hi, this is Elena and I'm going to be presenti...",大家好，我是Elena，我将向大家介绍我们的工作——检测西班牙语中的未同化借词：注释语料库和...
1,dev_2,dev,data/raw/acl6060_en_zh/dev/segmented_wavs/gold...,So we're going to be covering what lexical bor...,我们将讨论什么是词汇借用、我们提出的任务、我们已经发布的数据集，以及我们探索的一些模型。
2,dev_3,dev,data/raw/acl6060_en_zh/dev/segmented_wavs/gold...,"But to begin with, what is lexical borrowing a...",首先，什么是词汇借用，为什么它作为一个自然语言处理任务很重要？
3,dev_4,dev,data/raw/acl6060_en_zh/dev/segmented_wavs/gold...,"Well, lexical borrowing is basically the incor...",总的来说，词汇借用是将一种语言的单词并入另一种语言中。
4,dev_5,dev,data/raw/acl6060_en_zh/dev/segmented_wavs/gold...,"For instance, in Spanish we use words that com...",例如，在西班牙语中，我们会使用来自英语的单词。


In [ ]:
df["audio_path"] = df["audio_path"].apply(lambda p: str((PROJECT_ROOT / p).resolve()))
print(df["audio_path"].iloc[0])
print(Path(df["audio_path"].iloc[0]).exists())

In [ ]:
from pathlib import Path
import soundfile as sf
import torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
DEVICE = "cuda:0"

row = df.iloc[0]
audio_path = Path(row["audio_path"])

processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

audio, sr = sf.read(str(audio_path))
if audio.ndim > 1:
    audio = audio.mean(axis=1)

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio_url": str(audio_path)},
            {"type": "text", "text": "Translate this English speech into Chinese."},
        ],
    }
]

text = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=False
)

inputs = processor(
    text=text,
    audios=[audio],
    sampling_rate=sr,
    return_tensors="pt",
)

inputs = {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

pred = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0].strip()

print("ID:", row["id"])
print("REF:", row["target_zh"])
print("PRED:", pred)

In [2]:
from pathlib import Path
import soundfile as sf
import torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
DEVICE = "cuda:0"

row = df.iloc[0]
audio_path = Path(row["audio_path"])

processor = AutoProcessor.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model.eval()

audio, sr = sf.read(str(audio_path))
if audio.ndim > 1:
    audio = audio.mean(axis=1)

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio_url": str(audio_path)},
            {"type": "text", "text": "Translate this English speech into Chinese."},
        ],
    }
]

text = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=False
)

inputs = processor(
    text=text,
    audios=[audio],
    sampling_rate=sr,
    return_tensors="pt",
)

inputs = {k: v.to(DEVICE) if hasattr(v, "to") else v for k, v in inputs.items()}

with torch.no_grad():
    generated_ids = model.generate(**inputs, max_new_tokens=128)

generated_ids = generated_ids[:, inputs["input_ids"].size(1):]

pred = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)[0].strip()

print("ID:", row["id"])
print("REF:", row["target_zh"])
print("PRED:", pred)

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

LibsndfileError: Error opening 'data/raw/acl6060_en_zh/dev/segmented_wavs/gold/sent_1.wav': System error.